# lex.uz RAG: bge-m3 embedding (Colab GPU)

Model va sozlamalar Mac'dagi bilan bir xil: `BAAI/bge-m3`, CLS pooling, L2 normalizatsiya, **fp32** (sifat o'zgarmaydi).

Tartib:
1. Yuqori menyu: **Runtime → Change runtime type → T4 GPU** → Save.
2. Hujayralarni tepadan pastga ketma-ket ishga tushiring (har birida ▶ yoki Shift+Enter).
3. 2-hujayra fayl so'raydi: Mac'dan `data/colab/embed_input.jsonl.gz` ni tanlang.
4. Oxirida `vectors.npz` avtomatik yuklab olinadi. Uni Mac'dagi `lexuz-rag/data/colab/` papkasiga qo'ying.

In [ ]:
# 1. GPU borligini tekshirish
import torch, subprocess
assert torch.cuda.is_available(), "GPU yoq! Runtime -> Change runtime type -> T4 GPU ni tanlang"
print(torch.cuda.get_device_name(0))
!pip -q install sentence-transformers  # Colab da odatda bor

In [ ]:
# 2. Faylni yuklash (embed_input.jsonl.gz)
import os
from google.colab import files
if not os.path.exists("embed_input.jsonl.gz"):
    files.upload()
!ls -la embed_input.jsonl.gz

In [ ]:
# 3. Chunklarni o'qish
import gzip, json
rows = [json.loads(l) for l in gzip.open("embed_input.jsonl.gz", "rt", encoding="utf-8")]
print(len(rows), "chunk")

In [ ]:
# 4. Model (fp32) va sozlamalar tekshiruvi
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-m3", device="cuda")   # fp32 - standart
model.max_seq_length = 1024                                 # chunklar <= 800 token
print(model)  # Transformer -> Pooling(cls) -> Normalize bo'lishi kerak
assert any("Normalize" in type(m).__name__ for m in model), "Normalize moduli yo'q"
pool = [m for m in model if type(m).__name__ == "Pooling"][0]
# sentence-transformers versiyasiga bog'liq bo'lmagan tekshiruv
assert "cls" in str(pool).lower(), pool
print("OK: CLS pooling + Normalize, dtype:", next(model.parameters()).dtype)

In [ ]:
# 5. Embedding (T4 da taxminan 30-60 daqiqa). Uzunlik bo'yicha saralanadi - padding kam, natija o'zgarmaydi.
import numpy as np, time
order = sorted(range(len(rows)), key=lambda i: len(rows[i]["text"]))
texts = [rows[i]["text"] for i in order]
t0 = time.time()
emb = model.encode(texts, batch_size=32, normalize_embeddings=True, convert_to_numpy=True,
                   show_progress_bar=True)
vecs = np.zeros((len(rows), emb.shape[1]), dtype=np.float32)
vecs[order] = emb
print(vecs.shape, vecs.dtype, f"{time.time() - t0:.0f}s")

In [ ]:
# 6. Saqlash va yuklab olish
np.savez("vectors.npz", ids=np.array([r["id"] for r in rows]), hashes=np.array([r["h"] for r in rows]),
         vectors=vecs, model=np.array("BAAI/bge-m3"))
!ls -la vectors.npz
files.download("vectors.npz")